In [1]:
# !pip install openai-whisper 
# !pip install pydub
# !pip install ffmpeg-python

In [2]:
import os
import torch
import whisper
from pydub import AudioSegment
from transformers import T5Tokenizer, T5ForConditionalGeneration
import re # Importa a biblioteca de expressões regulares para o pós-processamento

In [3]:
# ===============================
# FUNÇÃO: Gerar resumo com T5
# ===============================
def summarize_text(text):
    print("\nIniciando sumarização...")
    tokenizer = T5Tokenizer.from_pretrained(T5_MODEL)
    summarizer = T5ForConditionalGeneration.from_pretrained(T5_MODEL).to(device)

    input_text = "resuma: " + text
    inputs = tokenizer.encode(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    summary_ids = summarizer.generate(inputs,  min_length=50, length_penalty=2.0, num_beams=4)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    return summary

In [4]:
# ===============================
# FUNÇÃO PRINCIPAL
# ===============================
def process_audio_files(audio_files):
    whisper_model = whisper.load_model(WHISPER_MODEL).to(device)

    for audio_file in audio_files:
        wav_path = convert_to_wav(audio_file)
        chunks = split_audio(wav_path)

        full_transcription = ""
        for chunk in chunks:
            text = transcribe_audio(chunk, whisper_model)
            full_transcription += text + " "

        print("\n--- TRANSCRIÇÃO COMPLETA ---\n")
        print(full_transcription)

        summary = summarize_text(full_transcription)
        print("\n--- RESUMO ---\n")
        print(summary)

In [5]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:


import os
import torch
import whisper
from pydub import AudioSegment
from transformers import T5Tokenizer, T5ForConditionalGeneration
import re # Importa a biblioteca de expressões regulares para o pós-processamento


# In[2]:


# ===============================
# CONFIGURAÇÕES GERAIS
# ===============================

# Tamanho máximo dos pedaços em milissegundos (5 minutos)
CHUNK_LENGTH_MS = 5 * 60 * 1000

# Modelo de transcrição Whisper
WHISPER_MODEL = "base"  # pode trocar por "small", "medium" ou "large"

# Modelo de sumarização
T5_MODEL = "unicamp-dl/ptt5-base-portuguese-vocab"

device = "cuda" if torch.cuda.is_available() else "cpu"


# In[3]:


# ===============================
# FUNÇÃO: Converter para WAV
# ===============================
def convert_to_wav(input_path):
    print(f"\nConvertendo '{input_path}' para WAV...")
    audio = AudioSegment.from_file(input_path)
    wav_path = os.path.splitext(input_path)[0] + "_temp.wav"
    audio.export(wav_path, format="wav")
    print(f"Conversão concluída: '{wav_path}'")
    return wav_path


# In[4]:


# ===============================
# FUNÇÃO: Dividir o áudio em pedaços
# ===============================
def split_audio(file_path, chunk_length=CHUNK_LENGTH_MS):
    print(f"\nDividindo áudio '{file_path}' em partes menores...")
    audio = AudioSegment.from_wav(file_path)
    chunks = []
    base_name = os.path.splitext(file_path)[0]
    for i in range(0, len(audio), chunk_length):
        chunk = audio[i:i + chunk_length]
        chunk_path = f"{base_name}_chunk_{i//chunk_length}.wav"
        chunk.export(chunk_path, format="wav")
        chunks.append(chunk_path)
    print(f"Total de {len(chunks)} partes geradas.")
    return chunks


# In[5]:


# ===============================
# FUNÇÃO: Transcrever um áudio
# ===============================
def transcribe_audio(file_path, model):
    print(f"\nTranscrevendo '{file_path}'...")
    result = model.transcribe(file_path, fp16=False)
    return result["text"]


# In[6]:


# ===============================
# FUNÇÃO: Gerar resumo com T5 (AJUSTADA com loop de re-geração para finalização)
# ===============================
def summarize_text(text):
    print("\nIniciando sumarização com parâmetros dinâmicos e loop de re-geração...")
    tokenizer = T5Tokenizer.from_pretrained(T5_MODEL)
    summarizer = T5ForConditionalGeneration.from_pretrained(T5_MODEL).to(device)

    input_text_prefix = "resuma: "
    input_tokens = tokenizer.encode(input_text_prefix + text, return_tensors="pt", truncation=True, max_length=512)
    input_token_count = input_tokens.shape[1]

    # --- Defina seus parâmetros dinâmicos de comprimento do resumo ---
    SUMMARY_RATIO_MIN = 0.15
    SUMMARY_RATIO_MAX = 0.25

    ABSOLUTE_MIN_SUMMARY_LENGTH = 50
    ABSOLUTE_MAX_SUMMARY_LENGTH = 300

    # 1. Calcular o comprimento máximo dinâmico inicial
    dynamic_max_length_target = int(input_token_count * SUMMARY_RATIO_MAX)
    initial_effective_max_length = max(ABSOLUTE_MIN_SUMMARY_LENGTH, dynamic_max_length_target)
    initial_effective_max_length = min(ABSOLUTE_MAX_SUMMARY_LENGTH, initial_effective_max_length)

    # 2. Calcular o comprimento mínimo dinâmico inicial
    dynamic_min_length_target = int(input_token_count * SUMMARY_RATIO_MIN)
    initial_effective_min_length = max(ABSOLUTE_MIN_SUMMARY_LENGTH - 40, dynamic_min_length_target)
    initial_effective_min_length = max(10, initial_effective_min_length)

    # 3. Ajuste especial para textos de entrada muito curtos
    if input_token_count < ABSOLUTE_MIN_SUMMARY_LENGTH * 1.5:
        initial_effective_max_length = min(input_token_count + 10, ABSOLUTE_MIN_SUMMARY_LENGTH + 50)
        initial_effective_min_length = min(input_token_count - 10, initial_effective_max_length - 10)
        initial_effective_min_length = max(10, initial_effective_min_length)

    # 4. Garantir que max_length seja sempre maior que min_length
    if initial_effective_max_length < initial_effective_min_length:
        initial_effective_max_length = initial_effective_min_length + 10

    print(f"Gerando resumo para input de {input_token_count} tokens.")
    print(f"Parâmetros de geração iniciais: max_length={initial_effective_max_length}, min_length={initial_effective_min_length} tokens.")

    # --- Parâmetros para o loop de re-geração ---
    INCREMENT_STEP = 1  # Quantos tokens aumentar a cada tentativa
    MAX_RETRIES = 50      # Número máximo de tentativas de re-gerar
    # Limite máximo absoluto de tokens para o resumo, mesmo durante as tentativas de re-geração.
    # Isso evita que o resumo se torne excessivamente longo se o modelo não conseguir finalizar.
    REGENERATION_HARD_MAX_LENGTH = ABSOLUTE_MAX_SUMMARY_LENGTH + 100

    current_max_length = initial_effective_max_length
    current_min_length = initial_effective_min_length
    summary = ""
    retry_count = 0

    # Loop para gerar e verificar o resumo
    while retry_count <= MAX_RETRIES:
        #print(f"  Tentativa {retry_count + 1}: Gerando com max_length={current_max_length}, min_length={current_min_length}.")

        summary_ids = summarizer.generate(
            input_tokens.to(device),
            max_length=current_max_length,
            min_length=current_min_length,
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True # Ajuda o modelo a finalizar se encontrar um EOS token
        )
        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

        # Verifica se o resumo termina com um sinal de pontuação terminal (., !, ?)
        # Considera também aspas ou espaços após a pontuação.
        if re.search(r'[.?!]["\']?\s*$', summary.strip()):
            print("  Resumo gerado termina em frase completa. Sucesso!")
            break # Sai do loop, o resumo está bom

        #print(f"  Resumo gerado não termina em frase completa. Tentando novamente com max_length maior.")
        retry_count += 1

        # Condição de saída em caso de muitas tentativas ou resumo muito longo
        if retry_count > MAX_RETRIES or current_max_length >= REGENERATION_HARD_MAX_LENGTH:
            print(f"  Atingido o limite de tentativas ({MAX_RETRIES}) ou o comprimento máximo de regeneração ({REGENERATION_HARD_MAX_LENGTH} tokens).")
            # Se ainda não terminou em pontuação, adiciona reticências como último recurso para indicar interrupção.
            if not re.search(r'[.?!]["\']?\s*$', summary.strip()):
                 summary = summary.strip() + "..."
            break # Sai do loop

        # Incrementa os comprimentos para a próxima tentativa
        current_max_length += INCREMENT_STEP
        # Garante que min_length também aumenta, mas sempre mantendo uma margem para max_length
        current_min_length = min(current_max_length - 10, current_min_length + int(INCREMENT_STEP / 2))
        current_min_length = max(10, current_min_length) # Garante que não fique muito baixo

    return summary


# In[10]:


# ===============================
# FUNÇÃO PRINCIPAL
# ===============================
def process_audio_files(audio_files):
    whisper_model = whisper.load_model(WHISPER_MODEL).to(device)

    for audio_file in audio_files:
        temp_files_to_clean = [] # Lista para armazenar os arquivos temporários criados para este áudio

        try:
            wav_path = convert_to_wav(audio_file)
            temp_files_to_clean.append(wav_path) # Adiciona o arquivo WAV temporário à lista

            chunks = split_audio(wav_path)
            temp_files_to_clean.extend(chunks) # Adiciona todos os chunks à lista

            full_transcription = ""
            for chunk in chunks:
                text = transcribe_audio(chunk, whisper_model)
                full_transcription += text + " "

            print("\n--- TRANSCRIÇÃO COMPLETA ---\n")
            print(full_transcription)

            summary = summarize_text(full_transcription)
            print("\n--- RESUMO FINAL (COM FINALIZAÇÃO GARANTIDA) ---\n")
            print(summary)

        except Exception as e:
            print(f"Ocorreu um erro ao processar '{audio_file}': {e}")
        finally:
            # Limpa todos os arquivos temporários gerados para este áudio
            print(f"\nLimpando arquivos temporários para '{audio_file}'...")
            for f_path in temp_files_to_clean:
                if os.path.exists(f_path):
                    os.remove(f_path)
                    print(f"  - Removido: '{f_path}'")
                else:
                    print(f"  - Arquivo não encontrado (já removido ou não criado): '{f_path}'")


# In[11]:


# ===============================
# EXECUÇÃO
# ===============================
if __name__ == "__main__":
    # Exemplo: adicione aqui seus arquivos MP3/WAV
    arquivos = ['audio1.mp3', 'audio2.mp3','audio3.mp3']

    process_audio_files(arquivos)



Convertendo 'audio1.mp3' para WAV...
Conversão concluída: 'audio1_temp.wav'

Dividindo áudio 'audio1_temp.wav' em partes menores...
Total de 1 partes geradas.

Transcrevendo 'audio1_temp_chunk_0.wav'...

--- TRANSCRIÇÃO COMPLETA ---

 Não quero ser imperador, eu não entendo distas. Não quero governar-me com que estarem ninguém. Eu gostaria de ajudar todo mundo se possível, judeus, nativos, pretos e brancos. Nós todos queremos ajudar uns outros os seus humanos são assim. Queremos viver para felicidade e não para internecidade de todos. Não queremos odiar de pregar uns outros. Neste mundo, isto lugar para todos e até errica. E pode muito bem alimentar a todos. Os caminhos da vida podem ser livres e lindos. Mas nós perdemos este caminho. A ambição envenenou a alma dos homens. Erguei um muro de ódio ao retorno muito. Nós atiremos dentro da miseria e tão bem do ódio. Nós desenvolvemos a velocidade, mas nós deixamos em nós mesmos. As máquinas que nos torceram a mudança nos deixaram desampar

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Gerando resumo para input de 512 tokens.
Parâmetros de geração iniciais: max_length=128, min_length=76 tokens.
  Resumo gerado termina em frase completa. Sucesso!

--- RESUMO FINAL (COM FINALIZAÇÃO GARANTIDA) ---

resuma: Não quero ser imperador, eu não entendo distas. Não quero governar-me com que estarem ninguém. Eu gostaria de ajudar todo mundo se possível, judeus, nativos, pretos e brancos. Nós todos queremos ajudar uns outros os seus humanos são assim. Queremos viver para felicidade e não para internecidade de todos. Não queremos odiar de pregar uns outros. Neste mundo, isto lugar para todos e até errica. E pode muito bem alimentar a todos. Os caminhos da vida podem ser livres e lindos. Mas nós perdemos este caminho. A ambição envenenou a alma dos homens.

Limpando arquivos temporários para 'audio1.mp3'...
  - Removido: 'audio1_temp.wav'
  - Removido: 'audio1_temp_chunk_0.wav'

Convertendo 'audio2.mp3' para WAV...
Conversão concluída: 'audio2_temp.wav'

Dividindo áudio 'audio2_tem